In [1]:
import os

# Print the current working directory
print("Working directory:", os.getcwd())

# Check the data folder exists
if os.path.exists("../data"):
    print("✓ data/ folder found")
else:
    print("✗ data/ folder NOT found — make sure you opened the respiratory_ml folder in VS Code")

Working directory: /Users/minindu/respiratory_ml/notebooks
✓ data/ folder found


In [2]:
import requests
import os

# Where to save the files
SAVE_DIR = "../data"
BASE_URL = "https://physionet.org/files/bidmc/1.0.0/bidmc_csv/"

print("Starting download of BIDMC dataset...")
print("This will download ~10 MB of CSV files (53 patients × 3 files each)")
print("-" * 60)

downloaded = 0
skipped = 0
failed = []

for i in range(1, 54):  # patients 01 through 53
    patient_id = f"{i:02d}"  # turns 1 into "01", 12 into "12", etc.

    # The three file types we want per patient
    files_to_download = [
        f"bidmc_{patient_id}_Numerics.csv",  # SpO2, RR, HR — main data
        f"bidmc_{patient_id}_Fix.txt",        # age and gender
        f"bidmc_{patient_id}_Breaths.csv",    # breath annotations (optional but small)
    ]

    for filename in files_to_download:
        save_path = os.path.join(SAVE_DIR, filename)

        # Skip if already downloaded
        if os.path.exists(save_path):
            skipped += 1
            continue

        # Download the file
        url = BASE_URL + filename
        try:
            response = requests.get(url, timeout=30)
            response.raise_for_status()  # raises an error if download failed

            with open(save_path, "wb") as f:
                f.write(response.content)

            downloaded += 1
            print(f"  ✓ Downloaded {filename}")

        except Exception as e:
            failed.append(filename)
            print(f"  ✗ FAILED: {filename} — {e}")

print("-" * 60)
print(f"Done. Downloaded: {downloaded}  |  Skipped (already existed): {skipped}")

if failed:
    print(f"\nThese files failed — try re-running the cell:")
    for f in failed:
        print(f"  {f}")
else:
    print("All files downloaded successfully!")

Starting download of BIDMC dataset...
This will download ~10 MB of CSV files (53 patients × 3 files each)
------------------------------------------------------------
  ✓ Downloaded bidmc_01_Numerics.csv
  ✓ Downloaded bidmc_01_Fix.txt
  ✓ Downloaded bidmc_01_Breaths.csv
  ✓ Downloaded bidmc_02_Numerics.csv
  ✓ Downloaded bidmc_02_Fix.txt
  ✓ Downloaded bidmc_02_Breaths.csv
  ✓ Downloaded bidmc_03_Numerics.csv
  ✓ Downloaded bidmc_03_Fix.txt
  ✓ Downloaded bidmc_03_Breaths.csv
  ✓ Downloaded bidmc_04_Numerics.csv
  ✓ Downloaded bidmc_04_Fix.txt
  ✓ Downloaded bidmc_04_Breaths.csv
  ✓ Downloaded bidmc_05_Numerics.csv
  ✓ Downloaded bidmc_05_Fix.txt
  ✓ Downloaded bidmc_05_Breaths.csv
  ✓ Downloaded bidmc_06_Numerics.csv
  ✓ Downloaded bidmc_06_Fix.txt
  ✓ Downloaded bidmc_06_Breaths.csv
  ✓ Downloaded bidmc_07_Numerics.csv
  ✓ Downloaded bidmc_07_Fix.txt
  ✓ Downloaded bidmc_07_Breaths.csv
  ✓ Downloaded bidmc_08_Numerics.csv
  ✓ Downloaded bidmc_08_Fix.txt
  ✓ Downloaded bidmc_08_Breat

In [3]:
import os

DATA_DIR = "../data"

# Count what we have
numerics_files = sorted([f for f in os.listdir(DATA_DIR) if f.endswith("_Numerics.csv")])
fix_files      = sorted([f for f in os.listdir(DATA_DIR) if f.endswith("_Fix.txt")])

print(f"Numerics files found: {len(numerics_files)}  (expected 53)")
print(f"Fix files found:      {len(fix_files)}  (expected 53)")

# Show the first few so you can see the naming pattern
print("\nFirst 5 Numerics files:")
for f in numerics_files[:5]:
    size_kb = os.path.getsize(os.path.join(DATA_DIR, f)) / 1024
    print(f"  {f}  ({size_kb:.1f} KB)")

Numerics files found: 53  (expected 53)
Fix files found:      53  (expected 53)

First 5 Numerics files:
  bidmc_01_Numerics.csv  (7.5 KB)
  bidmc_02_Numerics.csv  (7.9 KB)
  bidmc_03_Numerics.csv  (7.4 KB)
  bidmc_04_Numerics.csv  (7.4 KB)
  bidmc_05_Numerics.csv  (7.7 KB)


In [4]:
import pandas as pd

DATA_DIR = "../data"

# ── Look at the Numerics CSV ──
print("=" * 50)
print("NUMERICS FILE (bidmc_01_Numerics.csv)")
print("=" * 50)

df = pd.read_csv(os.path.join(DATA_DIR, "bidmc_01_Numerics.csv"))

# Strip whitespace from column names — PhysioNet files sometimes
# have a leading space in column names like " HR" instead of "HR"
df.columns = df.columns.str.strip()

print(f"\nShape: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"\nColumn names: {list(df.columns)}")
print(f"\nFirst 10 rows:")
print(df.head(10).to_string())
print(f"\nBasic statistics:")
print(df.describe().round(2).to_string())
print(f"\nMissing values per column:")
print(df.isnull().sum())

NUMERICS FILE (bidmc_01_Numerics.csv)

Shape: 481 rows × 5 columns

Column names: ['Time [s]', 'HR', 'PULSE', 'RESP', 'SpO2']

First 10 rows:
   Time [s]  HR  PULSE  RESP  SpO2
0         0  94   93.0    25  97.0
1         1  94   93.0    25  97.0
2         2  94   93.0    25  97.0
3         3  92   93.0    26  97.0
4         4  93   93.0    26  97.0
5         5  93   93.0    26  97.0
6         6  94   94.0    26  97.0
7         7  94   94.0    26  97.0
8         8  94   94.0    26  97.0
9         9  94   94.0    26  97.0

Basic statistics:
       Time [s]      HR   PULSE    RESP    SpO2
count     481.0  481.00  468.00  481.00  468.00
mean      240.0   91.32   91.01   21.44   96.92
std       139.0    1.01    1.14    1.30    0.36
min         0.0   88.00   89.00   20.00   96.00
25%       120.0   91.00   90.00   21.00   97.00
50%       240.0   91.00   91.00   21.00   97.00
75%       360.0   92.00   91.00   22.00   97.00
max       480.0   95.00   94.00   26.00   98.00

Missing values per co

In [5]:
# ── Look at the Fix.txt ──
print("\n" + "=" * 50)
print("FIX FILE (bidmc_01_Fix.txt)")
print("=" * 50)

with open(os.path.join(DATA_DIR, "bidmc_01_Fix.txt"), "r") as f:
    print(f.read())


FIX FILE (bidmc_01_Fix.txt)
bidmc_01
Signals: RESP; PLETH; V; AVR; II
Signals sampling frequency: 125 Hz
Numerics: HR; PULSE; RESP; SpO2
Numerics sampling frequency: 1 Hz
Age: 88
Gender: M
Location: micu
MIMIC II matched wdb ID: s01182
MIMIC II data segment: 2688-03-25-23-14
Original overall header file: http://physionet.org/physiobank/database/mimic2wdb/matched/s01182/s01182-2688-03-25-23-14.hea
Original signals header file: http://physionet.org/physiobank/database/mimic2wdb/matched/s01182/3017887_0001.hea
Original signals data file: http://physionet.org/physiobank/database/mimic2wdb/matched/s01182/3017887_0001.dat
Original numerics header file: http://physionet.org/physiobank/database/mimic2wdb/matched/s01182/s01182-2688-03-25-23-14n.hea
Original numerics data file: http://physionet.org/physiobank/database/mimic2wdb/matched/s01182/3017887n.dat

